## Inference Pipeline


In [1]:
import os
import sys

sys.path.append(os.path.abspath(".."))

In [4]:
import joblib
import pandas as pd
from src.model import load_model

### Load Model

In [6]:
bundle = load_model("../models/bundle.joblib")
pipe=bundle['model']

### Input Schema

In [7]:
expected_cols = list(pipe.feature_names_in_)

len(expected_cols), expected_cols[:10]

(164,
 ['SK_ID_CURR',
  'NAME_CONTRACT_TYPE',
  'CODE_GENDER',
  'FLAG_OWN_CAR',
  'FLAG_OWN_REALTY',
  'CNT_CHILDREN',
  'AMT_INCOME_TOTAL',
  'AMT_CREDIT',
  'AMT_ANNUITY',
  'AMT_GOODS_PRICE'])

The model expects exactly 164 features. Input must match schema exactly.

### Sample Input 

In [12]:
encoder = pipe.named_steps["prep"].named_transformers_["cat"]
categories = encoder.categories_
cat_cols = pipe.named_steps["prep"].transformers_[0][2]
cat_map = dict(zip(cat_cols, categories))

sample = {}

for col in pipe.feature_names_in_:
    if col in cat_cols:
        sample[col] = cat_map[col][0]
    else:
        sample[col] = 0

sample.update({
    "AMT_INCOME_TOTAL": 2000000.0,
    "AMT_CREDIT": 500000.0,
    "EXT_SOURCE_2": 0.4,
    "EXT_SOURCE_3": 0.5,
    "CODE_GENDER": 'M'
})

### SImulate Inference

In [13]:
df = pd.DataFrame([sample])
df = df.reindex(columns=pipe.feature_names_in_, fill_value=0)

prob = pipe.predict_proba(df)[:,1][0]
prob

np.float64(0.6276366285785324)

### Risk Band + Decision Logic

In [14]:
def risk_band(prob):
    if prob < 0.2:
        return "Low Risk"
    elif prob < 0.5:
        return "Medium Risk"
    else:
        return "High Risk"

def decision(prob):
    if prob < 0.3:
        return "Approve"
    elif prob < 0.6:
        return "Review"
    else:
        return "Reject"

risk_band(prob), decision(prob)

('High Risk', 'Reject')

### Final Inference Function

In [15]:
def inference(data: dict):
    df = pd.DataFrame([data])
    df = df.reindex(columns=pipe.feature_names_in_, fill_value=0)

    for col in cat_cols:
        df[col] = df[col].astype(str)
    
    prob = pipe.predict_proba(df)[:,1][0]

    return {
        "probability": float(prob),
        "risk_band": risk_band(prob),
        "decision": decision(prob)
    }

### Test Multiple Case

In [16]:
test_cases = [
    {"AMT_INCOME_TOTAL": 100000, "EXT_SOURCE_2": 0.2},
    {"AMT_INCOME_TOTAL": 500000, "EXT_SOURCE_2": 0.8},
    {"AMT_CREDIT": 350000, "EXT_SOURCE_2": 0.78, "AMT_INCOME_TOTAL": 100000}
]

for t in test_cases:
    print(inference(t))

{'probability': 0.7120315637360694, 'risk_band': 'High Risk', 'decision': 'Reject'}
{'probability': 0.26557981900550753, 'risk_band': 'Medium Risk', 'decision': 'Approve'}
{'probability': 0.3096609650013698, 'risk_band': 'Medium Risk', 'decision': 'Review'}


c:\Users\ASUS\OneDrive\Desktop\AI\RiskForge\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\ASUS\OneDrive\Desktop\AI\RiskForge\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\ASUS\OneDrive\Desktop\AI\RiskForge\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
